# Answer Ladder: does a coding agent act on what it is given?

Companion notebook to the paper *Can Once, Cannot Reliably: Given the answer, does a small coding agent act on it?* (Gemma 4 Developer Agent paper track).

Each of 77 official competition tasks is given to the agent at five rungs of help: nothing, the reference lines, the reference patch, the patch as ready-made edit commands, and the patch with a one-call apply tool. Rungs 3-5 contain the answer, so the gap to 100% measures **execution**. This notebook reproduces the paper's ladder and code-graph tables from the released sessions (CPU, under a minute) and shows how to score your own agent.

In [ ]:
import json, os, sys, glob, subprocess, tarfile, collections
from pathlib import Path
# On Kaggle the dataset is mounted under /kaggle/input; locally, set ANSWER_LADDER to the release folder.
def find_root():
    """The release folder: ANSWER_LADDER, else the dataset under /kaggle/input (mount depth varies), else here."""
    if os.environ.get("ANSWER_LADDER"):
        return Path(os.environ["ANSWER_LADDER"])
    for depth in range(1, 6):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + "scripts/score.py")
        if hits:
            return Path(hits[0]).parent.parent
    return Path(".")
ROOT = find_root()
print(ROOT, sorted(p.name for p in ROOT.iterdir()))

## 1. The ladder and the graph arms, reproduced from the released sessions
Every number below is recomputed from `sessions/*.tar.gz` (one `task_results.jsonl` per session) by `scripts/score.py`: share of tries solved, solves per pass, tasks solved at least once and in every try, and paired 95% task-bootstrap intervals between adjacent rungs.

In [ ]:
out = subprocess.run([sys.executable, str(ROOT / "scripts/score.py"), "--release", str(ROOT / "sessions"),
                      "--ids", str(ROOT / "tasks/ids_77.txt")], capture_output=True, text=True)
print(out.stdout or out.stderr)

## 2. Figure: tries solved and tasks solved every time, per rung

In [ ]:
import matplotlib.pyplot as plt
RUNGS = ["control", "reference lines", "reference patch", "edit commands", "apply tool"]
blocks = out.stdout.split("\n\n")
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
for blk in blocks:
    lines = blk.strip().splitlines()
    if not lines or "Graph" in lines[0] or not any(l.strip().startswith("apply tool") for l in lines):
        continue
    name = lines[0].split(":")[0]
    rows = {l[2:30].strip(): l[30:].split() for l in lines[2:] if l[2:30].strip() in RUNGS}
    a.plot(range(5), [float(rows[r][1].rstrip("%")) for r in RUNGS], marker="o", label=name)
    b.plot(range(5), [int(rows[r][4]) for r in RUNGS], marker="o", label=name)
for ax, t in ((a, "Tries solved (%)"), (b, "Tasks solved in all 3 tries (of 77)")):
    ax.set_xticks(range(5), ["Nothing", "Lines", "Patch", "Commands", "Apply tool"]); ax.set_title(t); ax.grid(alpha=.3)
a.legend(fontsize=8); plt.tight_layout(); plt.show()

## 3. Did the agent use the one-call apply tool?
Each session ships `tool_calls.jsonl` (name and arguments of every model tool call). A try *invoked* the tool if any call names `apply_fix`; the tool *applied* the fix if the session events contain `FIX APPLIED`.

In [ ]:
# Apply-tool use, from the released sessions (archives, or the unpacked folders Kaggle shows).
# E4B, both scaffolds (Section 4): never invoked / invoked, and whether FIX APPLIED; then the official loop per model.
for script in ("apply_tool_use.py", "tool_use_models.py"):
    out_tu = subprocess.run([sys.executable, str(ROOT / "scripts" / script), str(ROOT / "sessions")], capture_output=True, text=True)
    print(out_tu.stdout or out_tu.stderr)


## 4. Score your own agent
1. Run your agent with the competition harness (`swegemma`) on `tasks/ids_77.txt`, three tries per task, once per rung, with `TASKS_FILE` set to that rung's file in `tasks/<scaffold>/` (e.g. `tasks/official_loop/tasks_e4b_off_diff_v1.jsonl` for the patch rung).
2. For the apply-tool rung, add `apply_tool/swe_fixtool-1.0-py3-none-any.whl` to a copy of the task's wheels folder used only for that rung: the harness unpacks it into the sandbox and `python3 -m apply_fix <task id>` applies the saved reference patch.
3. Score the result folders:

In [ ]:
# python3 scripts/score.py --rung control=runs/ctrl --rung lines=runs/loc --rung patch=runs/diff --rung commands=runs/edits --rung apply=runs/apply
print(subprocess.run([sys.executable, str(ROOT / "scripts/score.py"), "--help"], capture_output=True, text=True).stdout)